## This QNA ChatBot is for Charter Communications on Wikipedia

1] Import the required libraries

In [1]:
import os
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

d:\shree\project_files\samples\langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


2] Load the model

In [34]:
from dotenv import load_dotenv
load_dotenv()

#Using the model from Groq
groq_api = os.getenv("GROQ_API_KEY")

#ChatGroq is a class provided by the langchain_groq package that allows you to interact with Groq's language models. By initializing an instance of ChatGroq with your API key and specifying the model you want to use, you can generate responses based on the input you provide. In this case, we're using the "llama-3.1-8b-instant" model, which is a powerful language model capable of understanding and generating human-like text.
llm = ChatGroq(groq_api_key=groq_api,model="llama-3.1-8b-instant") # Initialize the language model (LLM) that will generate responses based on retrieved information. Here, we use Groq's LLM, but this could be swapped out for any compatible model.

3] Load the Embeddings

Emebdding converts the human language words into vectors(numbers which computer can understand), such that it captures the semantic meaning and context of the word.

These act as a vocabulary for any given model

In [35]:
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3588.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4] Load the website used for qna

In [36]:
import bs4

custom_strainer = bs4.SoupStrainer(id="mw-content-text")
# custom_strainer = bs4.SoupStrainer(class_=["typestack-title-3", "text"])
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
loader = WebBaseLoader(
    web_path="https://en.wikipedia.org/wiki/Charter_Communications",
    requests_kwargs={"headers": headers},
    bs_kwargs={"parse_only": custom_strainer},)

docs = loader.load()
# docs[0].page_content[:500]


5] Create Retriver

Retriver is used to pull the most relevant information from the vector store

In [37]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

6] Prompt Template

In [38]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

7] Create a chat history

In [51]:
cont_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

cont_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", cont_q_system_prompt),
         MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
       
    ]
)


history_retiriver = create_history_aware_retriever(llm=llm, retriever=retriever, prompt=cont_q_prompt)

8] Create a RAG Chain

In [52]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)
rag_chain = create_retrieval_chain(retriever=history_retiriver,combine_docs_chain=qa_chain)


9] Store the session history

In [53]:
store = {

}

def get_session_history(Session_id) ->BaseChatMessageHistory:
    if Session_id not in store:
        store[Session_id] = ChatMessageHistory()
    return store[Session_id]



10] Create a Runnable with rag chain

In [54]:
conv_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"

)


def config_session(Session_id):
    return {"configurable":{"session_id": Session_id}}


def ask_message(session_id: str, question: str) -> str:

    response = conv_rag_chain.invoke(
        {"input": question},
        config=config_session(session_id)
    )
    
    return response['answer']

In [55]:
ask_message("test_session01", "What is Charter Communications?")

'Charter Communications is an American telecommunications and mass media company with services branded as Spectrum. It is headquartered in Stamford, Connecticut. The company provides internet, TV, and phone services to over 32 million customers in 41 states.'

In [56]:
ask_message("test_session01", "1980–1992: Beginnings, what happend here")

'Charter Communications CATV systems was founded in 1980 by Charles H. Leonard in Barry County, Michigan. The original headquarters and offices were located at 1001 Payne Lake Road, Yankee Springs Township, Michigan. Leonard began a corporate partnership with Gary Wilcox and Gerry Kazma, leading to the merger of Spectrum Communications (Wilcox) with Charter Systems (1981–1983).'

In [57]:
ask_message("test_session01", "How much was the debt due to which they filed for bankruptcy?")

'Charter Communications had approximately $8 billion in debt that they aimed to reduce through their bankruptcy plan.'

In [58]:
ask_message("test_session01", "What about 2009?")

"In 2009, Charter Communications' debt was around $22 billion."

In [59]:
ask_message("test_session02", "How much was the debt due to which they filed for bankruptcy?")

'Charter Communications expected the financial restructuring to reduce its debt by $8 billion, as well as adding $3 billion of new investment, and refinancing other debt.'

In [60]:
ask_message("test_session01", "How much was FCC fine?")

'Charter Communications was fined $15 million by the FCC for violating 911 outage reporting rules.'

In [49]:
ask_message("test_session02", "How much was FCC fine?")

"I don't know."

In [61]:
ask_message("test_session03", "How much was FCC fine?")

'The FCC fined Charter Communications $15 million for failing to report 911 outages.'